<a href="https://colab.research.google.com/github/Sumit7504/Informal-Work-Zone-Perception-for-Autonomous-Vehicles/blob/main/automatedDrivingSystemsADS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import shutil
import torch
import urllib.request
import pandas as pd
from ultralytics import YOLO, YOLOWorld
from IPython.display import display, HTML
from google.colab import files as colab_files

# ==========================================
# 1. CONFIGURATION & DIRECT FILE UPLOAD
# ==========================================
video_path = "/content/sample_video.mp4"

# Force clean-up of any old default videos so we don't accidentally reuse old results
if os.path.exists(video_path):
    os.remove(video_path)
if os.path.exists("/content/video1.mp4"):
    os.rename("/content/video1.mp4", "/content/old_first_video_backup.mp4")
    print("Backed up old 'video1.mp4' to prevent conflicts.")

print("Please upload your second video file directly from your computer:")
uploaded = colab_files.upload()

# Find the uploaded file name
uploaded_filename = list(uploaded.keys())[0]
shutil.move(uploaded_filename, video_path)
print(f"\nSuccess! Your video '{uploaded_filename}' has been successfully loaded and saved to {video_path}.")

interval_sec = 5  # Extract 1 frame every 5 seconds

# Clean and recreate output directories so old 0-detection images are replaced
for folder in ["1_extracted_frames", "2_annotated_screenshots", "3_red_device_closeups", "4_red_only_full_screenshots"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

# ==========================================
# 2. LOAD AI MODELS & MULTI-SYNONYM VOCABULARY
# ==========================================
print("Loading AI Models...")
model_closed = YOLO("yolov8x.pt")             # Closed-set baseline (COCO)
model_open = YOLOWorld("yolov8x-worldv2.pt")  # Open-vocabulary detector

# Visual synonym prompts tailored to catch the exact circled items in your frames,
# mapped cleanly to the official Paper Taxonomy Classes for Table III:
prompt_mapping = {
    # --- Standard Devices ---
    "traffic cone": ("traffic cone", "Standard"),
    "orange cone": ("traffic cone", "Standard"),
    "stop sign": ("STOP and warning signs", "Standard"),
    "white sign board": ("STOP and warning signs", "Standard"),
    "no parking sign": ("STOP and warning signs", "Standard"),
    "warning sign board": ("STOP and warning signs", "Standard"),
    "road sign": ("STOP and warning signs", "Standard"),
    "signboard on post": ("STOP and warning signs", "Standard"),
    "red plastic barrier": ("plastic water-filled barrier", "Standard"),
    "water-filled barrier": ("plastic water-filled barrier", "Standard"),

    # --- Informal / Local / Construction Materials ---
    "concrete barrier": ("concrete barrier block", "Informal / Local"),
    "concrete block": ("concrete barrier block", "Informal / Local"),
    "grey concrete block": ("concrete barrier block", "Informal / Local"),
    "concrete road divider": ("concrete barrier block", "Informal / Local"),
    "jersey barrier": ("concrete barrier block", "Informal / Local"),
    "metal barricade": ("steel barricade", "Informal / Local"),
    "steel栅栁": ("steel barricade", "Informal / Local"),
    "caution tape": ("barricade tape", "Informal / Local"),
    "barricade tape": ("barricade tape", "Informal / Local"),
    "painted oil drum": ("painted metal drum", "Informal / Local"),
    "metal drum": ("painted metal drum", "Informal / Local"),
    "barrel": ("painted metal drum", "Informal / Local"),
    "green safety net": ("banner or mesh screen", "Informal / Local"),
    "green mesh tarp": ("banner or mesh screen", "Informal / Local"),
    "cloth banner": ("banner or mesh screen", "Informal / Local"),
    "poster sign": ("banner or mesh screen", "Informal / Local"),
    "steel rods": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "iron rods on ground": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "rebar bundle": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "metal rods": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "stacked metal plates": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "formwork stack": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "pile of wooden boards": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "construction material stack": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "pile of sand and gravel": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "mound of dirt": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
    "rubble pile": ("materials on carriageway (steel/formwork/debris)", "Informal / Local"),
}

search_prompts = list(prompt_mapping.keys())
model_open.set_classes(search_prompts)

# Official aggregated classes for Table III
table3_classes = [
    ("traffic cone", "Standard"),
    ("STOP and warning signs", "Standard"),
    ("plastic water-filled barrier", "Standard"),
    ("concrete barrier block", "Informal / Local"),
    ("steel barricade", "Informal / Local"),
    ("barricade tape", "Informal / Local"),
    ("painted metal drum", "Informal / Local"),
    ("banner or mesh screen", "Informal / Local"),
    ("materials on carriageway (steel/formwork/debris)", "Informal / Local")
]

road_user_classes = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

closed_stats = {
    "Person": 0, "Car": 0, "Truck": 0, "Bus": 0,
    "Motorcycle": 0, "Bicycle": 0, "STOP sign": 0, "Implausible labels": 0
}
open_vocab_counts = {cls_name: {"count": 0, "conf_sum": 0.0} for cls_name, _ in table3_classes}

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = max(1, (boxA[2] - boxA[0]) * (boxA[3] - boxA[1]))
    boxBArea = max(1, (boxB[2] - boxB[0]) * (boxB[3] - boxB[1]))
    return interArea / float(boxAArea + boxBArea - interArea)

# ==========================================
# 3. EXTRACT FRAMES & RUN HIGH-RES DETECTION
# ==========================================
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or fps != fps:
    fps = 30.0
frame_step = max(1, int(fps * interval_sec))

frame_idx = 0
saved_frame_count = 0
closeup_count = 0
red_frames_with_detections = 0

print(f"Sampling 1 frame every {interval_sec} seconds (every {frame_step} frames)...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_step == 0:
        saved_frame_count += 1
        raw_frame = frame.copy()
        annotated_frame = frame.copy() # Folder 2: Combined BLUE + RED
        red_only_frame = frame.copy()  # Folder 4: ONLY RED boxes on full frame

        # Save raw extracted frame (Folder 1)
        cv2.imwrite(f"1_extracted_frames/frame_{saved_frame_count:03d}.jpg", raw_frame)

        # ----------------------------------------------------
        # A. Run Closed-Set Detector (YOLOv8x, conf=0.25)
        # ----------------------------------------------------
        res_closed = model_closed(raw_frame, conf=0.25, verbose=False)[0]
        blue_vehicle_boxes = []

        for box in res_closed.boxes:
            cls_id = int(box.cls[0])
            cls_name = model_closed.names[cls_id]
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            if cls_name in road_user_classes:
                blue_vehicle_boxes.append((x1, y1, x2, y2))
                if cls_name == "person": closed_stats["Person"] += 1
                elif cls_name == "car": closed_stats["Car"] += 1
                elif cls_name == "truck": closed_stats["Truck"] += 1
                elif cls_name == "bus": closed_stats["Bus"] += 1
                elif cls_name == "motorcycle": closed_stats["Motorcycle"] += 1
                elif cls_name == "bicycle": closed_stats["Bicycle"] += 1

                # Draw BLUE box ONLY on annotated_frame
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 100, 0), 2)
                cv2.putText(annotated_frame, f"{cls_name} ({conf:.2f})", (x1, max(20, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 100, 0), 2)

            elif cls_name == "stop sign":
                closed_stats["STOP sign"] += 1
            elif cls_name != "traffic light":
                closed_stats["Implausible labels"] += 1

        # ----------------------------------------------------
        # B. Run Open-Vocabulary Detector (High-Res imgsz=1280, conf=0.04)
        # ----------------------------------------------------
        res_open = model_open.predict(
            raw_frame,
            conf=0.04,
            imgsz=1280,
            iou=0.4,
            agnostic_nms=True,
            verbose=False
        )[0]

        valid_red_detections = 0

        for box in res_open.boxes:
            cls_id = int(box.cls[0])
            raw_prompt = search_prompts[cls_id]
            canonical_cls, _ = prompt_mapping[raw_prompt]
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Filter out false positives where a vehicle/person is misread as a work-zone item
            if any(compute_iou((x1, y1, x2, y2), b_box) > 0.50 for b_box in blue_vehicle_boxes):
                continue

            # Ignore overly huge background boxes covering >45% of the entire screen
            h, w, _ = raw_frame.shape
            if ((x2 - x1) * (y2 - y1)) > 0.45 * (w * h):
                continue

            valid_red_detections += 1
            open_vocab_counts[canonical_cls]["count"] += 1
            open_vocab_counts[canonical_cls]["conf_sum"] += conf

            # Display the specific matched visual label + confidence
            short_label = raw_prompt if len(raw_prompt) <= 22 else canonical_cls[:22]
            label_text = f"{short_label} ({conf:.2f})"
            (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)

            # 1. Draw RED box on annotated_frame
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.rectangle(annotated_frame, (x1, max(0, y1 - th - 8)), (x1 + tw + 4, y1), (0, 0, 255), -1)
            cv2.putText(annotated_frame, label_text, (x1 + 2, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            # 2. Draw RED box on red_only_frame
            cv2.rectangle(red_only_frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.rectangle(red_only_frame, (x1, max(0, y1 - th - 8)), (x1 + tw + 4, y1), (0, 0, 255), -1)
            cv2.putText(red_only_frame, label_text, (x1 + 2, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            # 3. Save Close-Up Crop of Every RED Detection
            pad = 15
            cx1, cy1 = max(0, x1 - pad), max(0, y1 - pad)
            cx2, cy2 = min(w, x2 + pad), min(h, y2 + pad)
            crop = raw_frame[cy1:cy2, cx1:cx2].copy()

            if crop.size > 0:
                closeup_count += 1
                header_h = 35
                canvas = cv2.copyMakeBorder(crop, header_h, 0, 0, 0, cv2.BORDER_CONSTANT, value=(0, 0, 0))
                cv2.putText(canvas, f'Red Det: "{short_label}" ({conf:.2f})', (8, 24),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

                safe_cls_name = short_label.replace(" ", "_").replace("/", "_")
                closeup_filename = f"3_red_device_closeups/crop_{closeup_count:03d}_frame{saved_frame_count:03d}_{safe_cls_name}.jpg"
                cv2.imwrite(closeup_filename, canvas)

        if valid_red_detections > 0:
            red_frames_with_detections += 1

        # Add Header Banner to Combined Frame
        cv2.rectangle(annotated_frame, (10, 10), (510, 45), (255, 255, 255), -1)
        cv2.putText(annotated_frame, "BLUE: Road Users (Closed-Set) | RED: Work-Zone Devices (Open-Vocab)",
                    (18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 0, 0), 1)
        cv2.imwrite(f"2_annotated_screenshots/annotated_frame_{saved_frame_count:03d}.jpg", annotated_frame)

        # Add Header Banner to RED-Only Full Frame
        cv2.rectangle(red_only_frame, (10, 10), (490, 45), (255, 255, 255), -1)
        cv2.putText(red_only_frame, f"RED: Work-Zone Devices Only (Open-Vocab) | Found: {valid_red_detections}",
                    (18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 0, 255), 1)
        cv2.imwrite(f"4_red_only_full_screenshots/red_full_frame_{saved_frame_count:03d}.jpg", red_only_frame)

    frame_idx += 1

cap.release()
print(f"\nProcessing Complete!")
print(f"- Extracted {saved_frame_count} frames (saved in '1_extracted_frames/')")
print(f"- Saved {saved_frame_count} combined BLUE+RED screenshots (saved in '2_annotated_screenshots/')")
print(f"- Saved {closeup_count} close-up crops of RED work-zone devices (saved in '3_red_device_closeups/')")
print(f"- Saved {saved_frame_count} full RED-only screenshots (saved in '4_red_only_full_screenshots/' — {red_frames_with_detections}/{saved_frame_count} frames had RED detections)\n")

# ==========================================
# 4. GENERATE OUTPUT TABLES (RESULTS FORMAT)
# ==========================================
print("="*80)
print("IV. RESULTS - A. TABLE I: TAXONOMY OF OBSERVED WORK-ZONE DEVICES")
print("="*80)
table1_data = [
    ["Traffic cone", "Specified (5.2.1)", "Cone 0.5-1 m, red-white bands", "Small at range; often displaced"],
    ["Water-filled plastic barrier", "Specified (5.2.8)", "Modular block, red-white", "Shape and colour vary by supplier"],
    ["Standard sign (STOP, warning, no parking)", "Specified (Sec. 5)", "Plate on post or stand", "Only STOP in generic label sets"],
    ["Concrete barrier block", "Specified form (5.2.7); local blocks vary", "Grey block", "Low contrast with road and dust"],
    ["Movable steel barricade", "Barricade (5.2.5); local agency design", "Open lattice frame", "See-through; thin members"],
    ["Barricade tape", "Allowed only with cones (5.2.1)", "Thin line, 0.5-1 m high", "Few pixels and LiDAR returns"],
    ["Painted metal drum", "Steel drums not permitted (5.2.4)", "Cylinder, irregular paint", "No consistent colour pattern"],
    ["Marshal hand signal", "Flagger with flag or paddle (9.2)", "Person and gesture", "Seen as a person; meaning not read"],
    ["Mesh or fabric screen", "Not provided for", "Mesh or fabric panel", "Semi-transparent; texture-like"],
    ["Printed flex banner", "Not provided for", "Text, non-standard units", "Must be read; mixed with advertising"],
    ["Materials on carriageway (steel/formwork/debris)", "Not a device", "Flat, stacked, or mounded", "Resembles road texture; few returns"]
]
display(pd.DataFrame(table1_data, columns=["Device", "IRC:SP:55 Status", "Form", "Perceptual Difficulty"]))

print("\n" + "="*80)
print(f"IV. RESULTS - B. TABLE II: CLOSED-SET BASELINE (YOLOv8x, COCO, conf=0.25; {saved_frame_count} FRAMES)")
print("="*80)
table2_data = [
    ["Person", str(closed_stats["Person"])],
    ["Car / truck / bus", f'{closed_stats["Car"]} / {closed_stats["Truck"]} / {closed_stats["Bus"]}'],
    ["Motorcycle / bicycle", f'{closed_stats["Motorcycle"]} / {closed_stats["Bicycle"]}'],
    ["STOP sign (only device class in COCO)", str(closed_stats["STOP sign"])],
    ["Other work-zone devices", "None (no class)"],
    ["Implausible labels (e.g., boat, bench, refrigerator)", str(closed_stats["Implausible labels"])]
]
display(pd.DataFrame(table2_data, columns=["Category", "Detections"]))

print("\n" + "="*80)
print("IV. RESULTS - C. TABLE III: OPEN-VOCABULARY DETECTION BY DEVICE CLASS (YOLO-World, High-Res)")
print("="*80)
table3_rows = []
std_total, inf_total = 0, 0

for cls_name, group in table3_classes:
    cnt = open_vocab_counts[cls_name]["count"]
    avg_conf = (open_vocab_counts[cls_name]["conf_sum"] / cnt) if cnt > 0 else 0.0
    if group == "Standard": std_total += cnt
    else: inf_total += cnt
    table3_rows.append([cls_name, group, cnt, f"{avg_conf:.2f}" if cnt > 0 else "0.00", "Pending Ground Truth"])

table3_rows.append(["All Standard / All Informal", "Summary", f"{std_total} / {inf_total}", "-", "-"])
display(pd.DataFrame(table3_rows, columns=["Device Class", "Group", "Detected Count (n)", "Mean Confidence", "Recall / Precision"]))

Please upload your second video file directly from your computer:


Saving report3_video.mp4 to report3_video.mp4

Success! Your video 'report3_video.mp4' has been successfully loaded and saved to /content/sample_video.mp4.
Loading AI Models...
Sampling 1 frame every 5 seconds (every 149 frames)...

Processing Complete!
- Extracted 17 frames (saved in '1_extracted_frames/')
- Saved 17 combined BLUE+RED screenshots (saved in '2_annotated_screenshots/')
- Saved 73 close-up crops of RED work-zone devices (saved in '3_red_device_closeups/')
- Saved 17 full RED-only screenshots (saved in '4_red_only_full_screenshots/' — 16/17 frames had RED detections)

IV. RESULTS - A. TABLE I: TAXONOMY OF OBSERVED WORK-ZONE DEVICES


,Device,IRC:SP:55 Status,Form,Perceptual Difficulty
0,Traffic cone,Specified (5.2.1),"Cone 0.5-1 m, red-white bands",Small at range; often displaced
1,Water-filled plastic barrier,Specified (5.2.8),"Modular block, red-white",Shape and colour vary by supplier
2,"Standard sign (STOP, warning, no parking)",Specified (Sec. 5),Plate on post or stand,Only STOP in generic label sets
3,Concrete barrier block,Specified form (5.2.7); local blocks vary,Grey block,Low contrast with road and dust
4,Movable steel barricade,Barricade (5.2.5); local agency design,Open lattice frame,See-through; thin members
5,Barricade tape,Allowed only with cones (5.2.1),"Thin line, 0.5-1 m high",Few pixels and LiDAR returns
6,Painted metal drum,Steel drums not permitted (5.2.4),"Cylinder, irregular paint",No consistent colour pattern
7,Marshal hand signal,Flagger with flag or paddle (9.2),Person and gesture,Seen as a person; meaning not read
8,Mesh or fabric screen,Not provided for,Mesh or fabric panel,Semi-transparent; texture-like
9,Printed flex banner,Not provided for,"Text, non-standard units",Must be read; mixed with advertising



IV. RESULTS - B. TABLE II: CLOSED-SET BASELINE (YOLOv8x, COCO, conf=0.25; 17 FRAMES)


,Category,Detections
0,Person,49
1,Car / truck / bus,63 / 41 / 20
2,Motorcycle / bicycle,7 / 1
3,STOP sign (only device class in COCO),3
4,Other work-zone devices,None (no class)
5,"Implausible labels (e.g., boat, bench, refrige...",14



IV. RESULTS - C. TABLE III: OPEN-VOCABULARY DETECTION BY DEVICE CLASS (YOLO-World, High-Res)


,Device Class,Group,Detected Count (n),Mean Confidence,Recall / Precision
0,traffic cone,Standard,18,0.12,Pending Ground Truth
1,STOP and warning signs,Standard,17,0.14,Pending Ground Truth
2,plastic water-filled barrier,Standard,0,0.00,Pending Ground Truth
3,concrete barrier block,Informal / Local,2,0.11,Pending Ground Truth
4,steel barricade,Informal / Local,0,0.00,Pending Ground Truth
5,barricade tape,Informal / Local,0,0.00,Pending Ground Truth
6,painted metal drum,Informal / Local,16,0.16,Pending Ground Truth
7,banner or mesh screen,Informal / Local,0,0.00,Pending Ground Truth
8,materials on carriageway (steel/formwork/debris),Informal / Local,20,0.10,Pending Ground Truth
9,All Standard / All Informal,Summary,35 / 38,-,-


In [ ]:
!pip install -q openpyxl

import os
import glob
import base64
import shutil
import pandas as pd
from google.colab import files

# ==========================================
# 1. CREATE MASTER REPORT FOLDER
# ==========================================
report_dir = "WorkZone_Complete_Project_Report"
tables_dir = os.path.join(report_dir, "0_Tables_and_Data")
os.makedirs(tables_dir, exist_ok=True)

# Reconstruct the DataFrames from your previous cell's results
df_t1 = pd.DataFrame(table1_data, columns=["Device", "IRC:SP:55 Status", "Form", "Perceptual Difficulty"])
df_t2 = pd.DataFrame(table2_data, columns=["Category", "Detections"])
df_t3 = pd.DataFrame(table3_rows, columns=["Device Class", "Group", "Detected Count (n)", "Mean Confidence", "Recall / Precision"])

# Save individual CSV files
df_t1.to_csv(os.path.join(tables_dir, "Table_I_Device_Taxonomy.csv"), index=False)
df_t2.to_csv(os.path.join(tables_dir, "Table_II_Closed_Set_Baseline.csv"), index=False)
df_t3.to_csv(os.path.join(tables_dir, "Table_III_Open_Vocabulary_Detection.csv"), index=False)

# Save all 3 tables into a single multi-sheet Excel workbook (.xlsx)
excel_path = os.path.join(tables_dir, "All_Research_Tables_Report.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_t1.to_excel(writer, sheet_name="Table_I_Taxonomy", index=False)
    df_t2.to_excel(writer, sheet_name="Table_II_Closed_Set", index=False)
    df_t3.to_excel(writer, sheet_name="Table_III_Open_Vocab", index=False)

print("Saved CSV and Excel tables successfully!")

# ==========================================
# 2. COPY ALL 4 IMAGE FOLDERS INTO REPORT
# ==========================================
image_folders = [
    "1_extracted_frames",
    "2_annotated_screenshots",
    "3_red_device_closeups",
    "4_red_only_full_screenshots"
]

for folder in image_folders:
    if os.path.exists(folder):
        dest = os.path.join(report_dir, folder)
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(folder, dest)

# ==========================================
# 3. GENERATE STANDALONE HTML VISUAL REPORT
# ==========================================
def img_to_base64(img_path):
    with open(img_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# Embed sample images into the HTML report so you can print-to-PDF directly in browser
combined_imgs = sorted(glob.glob("2_annotated_screenshots/*.jpg"))
red_full_imgs = sorted(glob.glob("4_red_only_full_screenshots/*.jpg"))
closeup_imgs = sorted(glob.glob("3_red_device_closeups/*.jpg"))

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Informal Work-Zone Perception - Evaluation Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 30px; color: #222; line-height: 1.5; }}
        h1 {{ color: #b71c1c; border-bottom: 2px solid #b71c1c; padding-bottom: 8px; }}
        h2 {{ color: #1565c0; margin-top: 35px; border-bottom: 1px solid #ddd; padding-bottom: 5px; }}
        table {{ border-collapse: collapse; width: 100%; margin: 15px 0; font-size: 14px; }}
        th, td {{ border: 1px solid #ccc; padding: 10px; text-align: left; }}
        th {{ background-color: #f4f4f4; font-weight: bold; }}
        tr:nth-child(even) {{ background-color: #fafafa; }}
        .gallery {{ display: flex; flex-wrap: wrap; gap: 15px; margin-top: 15px; }}
        .card {{ border: 1px solid #ddd; padding: 8px; border-radius: 6px; background: #fff; width: 47%; box-sizing: border-box; }}
        .card-small {{ border: 1px solid #ddd; padding: 6px; border-radius: 6px; background: #fff; width: 22%; box-sizing: border-box; text-align: center; }}
        .card img, .card-small img {{ width: 100%; height: auto; border-radius: 4px; }}
        .caption {{ font-size: 12px; color: #555; margin-top: 5px; font-weight: bold; }}
        @media print {{ .card {{ width: 45%; page-break-inside: avoid; }} }}
    </style>
</head>
<body>
    <h1>Reading the Informal Work Zone: Experimental Evaluation Report</h1>
    <p><strong>Total Sampled Frames Evaluated:</strong> {saved_frame_count} frames (1 frame every {interval_sec} seconds)<br>
    <strong>Total RED Work-Zone Device Crops Extracted:</strong> {closeup_count} objects</p>

    <h2>IV-A. Table I: Taxonomy of Observed Work-Zone Devices</h2>
    {df_t1.to_html(index=False)}

    <h2>IV-B. Table II: Closed-Set Baseline (YOLOv8x, COCO)</h2>
    {df_t2.to_html(index=False)}

    <h2>IV-C. Table III: Open-Vocabulary Detection by Device Class (YOLO-World)</h2>
    {df_t3.to_html(index=False)}

    <h2>Visual Results 1: Combined Detections (BLUE: Road Users | RED: Work-Zone Devices)</h2>
    <div class="gallery">
"""

for img_p in combined_imgs:
    b64 = img_to_base64(img_p)
    fname = os.path.basename(img_p)
    html_content += f'<div class="card"><img src="data:image/jpeg;base64,{b64}"><div class="caption">{fname}</div></div>'

html_content += """
    </div>
    <h2>Visual Results 2: Full Scene — RED Work-Zone Devices Only</h2>
    <div class="gallery">
"""

for img_p in red_full_imgs:
    b64 = img_to_base64(img_p)
    fname = os.path.basename(img_p)
    html_content += f'<div class="card"><img src="data:image/jpeg;base64,{b64}"><div class="caption">{fname}</div></div>'

html_content += """
    </div>
    <h2>Visual Results 3: Close-Up Crops of Detected RED Work-Zone Devices</h2>
    <div class="gallery">
"""

for img_p in closeup_imgs:
    b64 = img_to_base64(img_p)
    fname = os.path.basename(img_p)
    html_content += f'<div class="card-small"><img src="data:image/jpeg;base64,{b64}"><div class="caption">{fname}</div></div>'

html_content += """
    </div>
</body>
</html>
"""

html_report_path = os.path.join(report_dir, "Full_Visual_and_Table_Report.html")
with open(html_report_path, "w", encoding="utf-8") as f:
    f.write(html_content)

# ==========================================
# 4. ZIP EVERYTHING & TRIGGER DOWNLOAD
# ==========================================
zip_filename = "WorkZone_Complete_Project_Report.zip"
shutil.make_archive("WorkZone_Complete_Project_Report", 'zip', report_dir)

print(f"\n Packaging complete! Downloading '{zip_filename}' to your computer...")
files.download(zip_filename)

Saved CSV and Excel tables successfully!

 Packaging complete! Downloading 'WorkZone_Complete_Project_Report.zip' to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>